# Construct h5ad expression x patch object

In [1]:
import sys 
import os
from datetime import datetime
today = datetime.now().strftime("%Y-%m-%d")

import anndata as ad
import hdf5plugin
import squidpy as sq
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from spatialdata_io.readers.xenium import xenium_aligned_image
import spatialdata as sd
import rapids_singlecell as rsc

import warnings
warnings.filterwarnings('ignore', category=ad.ImplicitModificationWarning)

# Add repo path to sys path (allows to access scripts and metadata from repo)
repo_path,_ = os.path.split(os.path.split(os.getcwd())[0])
repo_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses'
sys.path.insert(1, repo_path) 
sys.path.insert(2, '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses/scripts')
sys.path.insert(3, '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts')

# Autoreload custom scripts
%load_ext autoreload
%autoreload 2

# Define paths
plots_path = f'{repo_path}/plots/'
data_path = f'{repo_path}/data/'
model_path = os.path.join(repo_path, 'models')
general_data_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/data/'

print('Dir for plots: {}'.format(plots_path))
print('Dir for data: {}'.format(data_path))

# Formatting
from matplotlib import font_manager
font_manager.fontManager.addfont("/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf")
plt.style.use('/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts/plotting/thyAgeing.mplstyle')

from plotting.utils import calc_figsize, thyAgeing_greys, thyAgeing_colors
from anno_levels import get_ct_levels

Dir for plots: /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses/plots/
Dir for data: /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses/data/


## Load HnE data

In [2]:
# Load integrated HnE data
he_adata = ad.read_h5ad(f'{data_path}/morphoFM/thyAging_CAM_morphoFM_mpp025_mf2_he_coral.h5ad')

niche_labels = pd.read_excel('/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses/data/morpho_niches/morpho_niches.xlsx', index_col=0)[['morpho_niche_l3', 'morpho_niche_l2', 'morpho_niche_l1', 'morpho_niche_keep']].reset_index(names='leiden_r2.0')
niche_labels['leiden_r2.0'] = niche_labels['leiden_r2.0'].astype(str)
niche_labels['leiden_r2.0_concat'] = niche_labels['leiden_r2.0'].astype(str) + '_' + niche_labels['morpho_niche_l3'].astype(str)
niche_labels['leiden_r2.0_concat'] = pd.Categorical(niche_labels['leiden_r2.0_concat'], categories = sorted(niche_labels['leiden_r2.0_concat'].unique(), key = lambda x: int(x.split('_')[0])))
he_adata.obs = he_adata.obs.reset_index(names='tile_id').merge(niche_labels, on='leiden_r2.0', how='left').set_index('tile_id')
he_adata.obs['morpho_niche_l3'] = pd.Categorical(he_adata.obs['morpho_niche_l3'], categories=['cortex_septal','cortex_outer','cortex_dense', 'cortex', 'cortex_residual',
                    'medulla', 'medulla_hc', 'medulla_residual', 
                    'adipose', 'adipose_edge',
                    'fibrosis', 'fibrosis_adipose',
                    'endothelium_vessel', 'endothelium_edge',])
he_adata.obs['morpho_niche_l2'] = pd.Categorical(he_adata.obs['morpho_niche_l2'], categories=['cortex_septal','cortex', 'medulla', 'medulla_hc', 'adipose', 'fibrosis_adipose', 'fibrosis', 'vessel',])

he_adata.obs.head()

,sample,x0,y0,x1,y1,cx_img,cy_img,tile_size,stride,tissue_fraction,...,has_xenium,source,leiden_r1.0,leiden_r0.5,leiden_r2.0,morpho_niche_l3,morpho_niche_l2,morpho_niche_l1,morpho_niche_keep,leiden_r2.0_concat
tile_id,,,,,,,,,,,,,,,,,,,,,
A70-THY-0-FFPE-1_tile_00000000,A70-THY-0-FFPE-1,26208,1792,26432,2016,26320.0,1904.0,224,224,0.242128,...,False,CAM,4,10,29,fibrosis_adipose,fibrosis_adipose,fibrosis_adipose,True,29_fibrosis_adipose
A70-THY-0-FFPE-1_tile_00000001,A70-THY-0-FFPE-1,26432,1792,26656,2016,26544.0,1904.0,224,224,0.549685,...,False,CAM,12,4,17,fibrosis_adipose,fibrosis_adipose,fibrosis_adipose,True,17_fibrosis_adipose
A70-THY-0-FFPE-1_tile_00000002,A70-THY-0-FFPE-1,26656,1792,26880,2016,26768.0,1904.0,224,224,0.416354,...,False,CAM,15,1,6,fibrosis,fibrosis,fibrosis,True,6_fibrosis
A70-THY-0-FFPE-1_tile_00000003,A70-THY-0-FFPE-1,26880,1792,27104,2016,26992.0,1904.0,224,224,0.915537,...,False,CAM,0,1,6,fibrosis,fibrosis,fibrosis,True,6_fibrosis
A70-THY-0-FFPE-1_tile_00000004,A70-THY-0-FFPE-1,27104,1792,27328,2016,27216.0,1904.0,224,224,0.570532,...,False,CAM,12,2,25,fibrosis_adipose,fibrosis_adipose,fibrosis_adipose,True,25_fibrosis_adipose


In [6]:
# Map HnE coordinates to microns
mpp = 0.25  
patch_px = 224
patch_um = patch_px * mpp  

he_adata.obs[['x0_um', 'y0_um', 'x1_um', 'y1_um']] = np.int16(he_adata.obs[['x0', 'y0', 'x1', 'y1']] * mpp)
he_adata.obs[['patch_col', 'patch_row']] = (he_adata.obs[['x0', 'y0']] // patch_px).astype(int) 
he_adata.obs.head()

,sample,x0,y0,x1,y1,cx_img,cy_img,tile_size,stride,tissue_fraction,...,morpho_niche_l2,morpho_niche_l1,morpho_niche_keep,leiden_r2.0_concat,x0_um,y0_um,x1_um,y1_um,patch_col,patch_row
tile_id,,,,,,,,,,,,,,,,,,,,,
A70-THY-0-FFPE-1_tile_00000000,A70-THY-0-FFPE-1,26208,1792,26432,2016,26320.0,1904.0,224,224,0.242128,...,fibrosis_adipose,fibrosis_adipose,True,29_fibrosis_adipose,6552,448,6608,504,117,8
A70-THY-0-FFPE-1_tile_00000001,A70-THY-0-FFPE-1,26432,1792,26656,2016,26544.0,1904.0,224,224,0.549685,...,fibrosis_adipose,fibrosis_adipose,True,17_fibrosis_adipose,6608,448,6664,504,118,8
A70-THY-0-FFPE-1_tile_00000002,A70-THY-0-FFPE-1,26656,1792,26880,2016,26768.0,1904.0,224,224,0.416354,...,fibrosis,fibrosis,True,6_fibrosis,6664,448,6720,504,119,8
A70-THY-0-FFPE-1_tile_00000003,A70-THY-0-FFPE-1,26880,1792,27104,2016,26992.0,1904.0,224,224,0.915537,...,fibrosis,fibrosis,True,6_fibrosis,6720,448,6776,504,120,8
A70-THY-0-FFPE-1_tile_00000004,A70-THY-0-FFPE-1,27104,1792,27328,2016,27216.0,1904.0,224,224,0.570532,...,fibrosis_adipose,fibrosis_adipose,True,25_fibrosis_adipose,6776,448,6832,504,121,8


## Inspect spatialdata

In [5]:
xenium_sdata = sd.read_zarr(f'{data_path}/xenium/A72-THY-0-FFPE-1_xenium.zarr')
xenium_sdata

/tmp/ipykernel_1936793/3227199502.py:1: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  xenium_sdata = sd.read_zarr(f'{data_path}/xenium/A72-THY-0-FFPE-1_xenium.zarr')
2026-08-09 12:16:49 | [INFO] root_attr: multiscales
2026-08-09 12:16:49 | [INFO] root_attr: omero
2026-08-09 12:16:49 | [INFO] root_attr: spatialdata_attrs
2026-08-09 12:16:49 | [INFO] datasets [{'coordinateTransformations': [{'scale': [1.0, 1.0, 1.0], 'type': 'scale'}], 'path': '0'}, {'coordinateTransformations': [{'scale': [1.0, 2.0, 2.0000413342702434], 'type': 'scale'}], 'path': '1'}, {'coordinateTransformations': [{'scale': [1.0, 4.0, 4.000248015873016], 'type': 'scale'}], 'path': '2'}, {'coordinateTransformations': [{'scale': [1.0, 8.0, 8.000496031746032], 'type': 'scale'}], 'path': '3'}, {'coordinateTransformations': [{'scale': [1.0, 16.0, 16.000992063492063], 'type': 'scale'}], 'path': '4'}]
2026-08-09

SpatialData object, with associated Zarr store: /nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/data/xenium/A72-THY-0-FFPE-1_xenium.zarr
├── Images
│     └── 'morphology_focus': DataTree[cyx] (5, 54336, 48387), (5, 27168, 24193), (5, 13584, 12096), (5, 6792, 6048), (5, 3396, 3024)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (54336, 48387), (27168, 24193), (13584, 12096), (6792, 6048), (3396, 3024)
│     └── 'nucleus_labels': DataTree[yx] (54336, 48387), (27168, 24193), (13584, 12096), (6792, 6048), (3396, 3024)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (1049932, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (1049932, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (1020533, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (1049932, 5001)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images),

## Map tx to patches

In [9]:
import numpy as np, pandas as pd, scipy.sparse as sp, anndata as ad, spatialdata as sd, os, glob

PIXEL_SIZE, HE_PIXEL_SIZE = 0.2125, 0.221
mpp, patch_px = 0.25, 224
scale = HE_PIXEL_SIZE / mpp
alignment_dir = f'{data_path}/xenium/he_alignments'
QV_MIN, KMAX = 20, 1 << 20   # KMAX bounds patch_row/col for key packing

# --- rows of the output: the tissue-mask patch grid -------------------------
patches = (he_adata.obs[['SampleID', 'patch_col', 'patch_row']]
           .reset_index(names='patchID')
           .drop_duplicates(['SampleID', 'patch_col', 'patch_row'])
           .reset_index(drop=True))

samples = pd.Index(sorted(patches['SampleID'].unique()))
patches['sample_code'] = samples.get_indexer(patches['SampleID'])

def patch_key(scode, col, row):
    return (np.int64(scode) * KMAX + row.astype(np.int64)) * KMAX + col.astype(np.int64)

_keys = patch_key(patches['sample_code'].values, patches['patch_col'].values,
                  patches['patch_row'].values)
_ord         = np.argsort(_keys)
_keys_sorted = _keys[_ord]
_row_of_key  = _ord.astype(np.int64)          # sorted key -> row in `patches`

def lookup_patch(q):
    """Vectorised (sample, row, col) key -> patch row index; -1 if off-tissue."""
    pos = np.clip(np.searchsorted(_keys_sorted, q), 0, len(_keys_sorted) - 1)
    hit = _keys_sorted[pos] == q
    out = np.full(q.shape, -1, dtype=np.int64)
    out[hit] = _row_of_key[pos[hit]]
    return out

# --- cols of the output: the gene panel -------------------------------------
panel = xenium_sdata['table'].var
genes = pd.Index(panel.index[panel['feature_types'] == 'Gene Expression'])
n_patch, n_gene = len(patches), len(genes)

print(f'Found {n_patch} tissue patches and {n_gene} genes in the Xenium data.')

Found 415178 tissue patches and 5001 genes in the Xenium data.


In [10]:
def apply_affine(M, xy):
    h = np.hstack([xy, np.ones((len(xy), 1))])
    return (h @ M.T)[:, :2]

xenium_zarrs = {os.path.basename(p).split('-')[0]: p
                for p in sorted(glob.glob(f'{data_path}/xenium/*-THY-*_xenium.zarr'))}

counts   = sp.csr_matrix((n_patch, n_gene), dtype=np.int32)
n_seen = n_dropped = 0

for donor, zarr_path in xenium_zarrs.items():
    sdata = sd.read_zarr(zarr_path)
    M     = pd.read_csv(f'{alignment_dir}/{donor}_alignment_files/matrix.csv', header=None).values
    Minv  = np.linalg.inv(M)                      # H&E -> Xenium, so invert
    scode = samples.get_loc(f'{donor}.ome')

    ddf = sdata['transcripts'][['x', 'y', 'feature_name', 'qv']]
    for i in range(ddf.npartitions):
        part = ddf.partitions[i].compute()
        part = part[part['qv'] >= QV_MIN]
        if not len(part):
            continue

        # gene codes: map the ~5k categories, not the 3M rows
        cat  = part['feature_name'].astype('category')
        cmap = genes.get_indexer(cat.cat.categories.astype(str))
        code = cat.cat.codes.to_numpy()
        g    = np.where(code >= 0, cmap[code], -1)

        # microns -> Xenium px -> H&E native px -> target-mpp px -> patch
        xy  = apply_affine(Minv, part[['x', 'y']].to_numpy(np.float64) / PIXEL_SIZE) * scale
        col = np.floor(xy[:, 0] / patch_px).astype(np.int64)
        row = np.floor(xy[:, 1] / patch_px).astype(np.int64)

        m = (g >= 0) & (col >= 0) & (row >= 0) & (col < KMAX) & (row < KMAX)
        p = lookup_patch(patch_key(scode, col[m], row[m]))
        ok = p >= 0

        # coo -> csr sums duplicates == the one-hot matmul, never materialised
        counts += sp.coo_matrix(
            (np.ones(ok.sum(), np.int32), (p[ok], g[m][ok])),
            shape=(n_patch, n_gene)).tocsr()

        n_seen    += len(part)
        n_dropped += len(part) - ok.sum()
    del sdata

print(f'{n_seen:,} transcripts (QV>={QV_MIN}); '
      f'{n_dropped/n_seen:.1%} fell outside tissue-mask patches')


/tmp/ipykernel_1936793/3622633330.py:12: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr(zarr_path)
2026-08-09 12:33:29 | [INFO] root_attr: multiscales
2026-08-09 12:33:29 | [INFO] root_attr: omero
2026-08-09 12:33:29 | [INFO] root_attr: spatialdata_attrs
2026-08-09 12:33:29 | [INFO] datasets [{'coordinateTransformations': [{'scale': [1.0, 1.0, 1.0], 'type': 'scale'}], 'path': '0'}, {'coordinateTransformations': [{'scale': [1.0, 2.0, 2.000036991824807], 'type': 'scale'}], 'path': '1'}, {'coordinateTransformations': [{'scale': [1.0, 4.0, 4.000221959159514], 'type': 'scale'}], 'path': '2'}, {'coordinateTransformations': [{'scale': [1.0, 8.00078323869199, 8.000443918319029], 'type': 'scale'}], 'path': '3'}, {'coordinateTransformations': [{'scale': [1.0, 16.004700352526438, 16.000887836638057], 'type': 'scale'}], 'path': '4'}]
2026-08-09 12:33:29 | [INFO] re

699,793,802 transcripts (QV>=20); 0.4% fell outside tissue-mask patches


In [ ]:
# Construct an AnnData object for the patch expression data
patch_expr = ad.AnnData(X=counts.tocsr(),
                        obs=patches.set_index('patchID'),
                        var=pd.DataFrame(index=genes))
patch_expr.obs['n_transcripts'] = np.asarray(patch_expr.X.sum(1)).ravel()
patch_expr.obs = patch_expr.obs.join(
    he_adata.obs[niche_labels.columns.difference(['leiden_r2.0']).tolist()], how='left')
patch_expr.obs['donor'] = patch_expr.obs['SampleID'].str.split('-').str[0]
patch_expr.var_names = genes

In [ ]:
# Save the patch expression data to an h5ad file with gzip compression
patch_expr.write_h5ad(f'{data_path}/morphoFM/thyAging_CAM_morphoFM_mpp025_mf2_patchExpr.h5ad', compression='gzip')
f'{data_path}/morphoFM/thyAging_CAM_morphoFM_mpp025_mf2_patchExpr.h5ad'

... storing 'morpho_niche_l1' as categorical
... storing 'donor' as categorical


'/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses/data//morphoFM/thyAging_CAM_morphoFM_mpp025_mf2_patchExpr.h5ad'